# Corrected late fusion and internal evaluation

**Scientific purpose.** Document fold-specific clinical/fusion logistic regression, verify the
exact eight-feature contract, and inspect corrected aggregate internal and statistical results.

**Runnability classification:** public aggregate reanalysis is runnable from a clean clone.
Patient-level refitting requires private OOF probabilities, held-out probabilities, clinical
variables, labels, and fold assignments.

The corrected primary analysis retains unresolved sex as missing and fits training-fold median
imputation. The historical non-male-to-zero path is a separate sensitivity analysis. W3 is a
standalone comparator and is excluded from full fusion.


In [ ]:
from pathlib import Path
import sys


def locate_repository(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


REPO_ROOT = locate_repository()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

from liver_tumor_pipeline.constants import CLASS_ORDER, FULL_FUSION_FEATURES
from liver_tumor_pipeline.fusion import build_full_fusion_features

expected_order = (
    "w4_p_HCC", "w4_p_ICC", "w4_p_CHCC",
    "w5_p_HCC", "w5_p_ICC", "w5_p_CHCC",
    "age", "sex",
)
if FULL_FUSION_FEATURES != expected_order or any(name.startswith("w3_") for name in expected_order):
    raise RuntimeError("Eight-feature full-fusion contract is inconsistent.")
pd.DataFrame({"position": range(1, 9), "feature": expected_order})


## Fold-specific corrected fusion definition

Each outer fold fits median imputation, standardization, and balanced multinomial logistic
regression using the other four OOF folds, then predicts its held-out fold. Five fitted models are
ensembled for the independent 56-patient evaluation cohort. Missing clinical values remain `NaN`
until the imputer is fitted on the corresponding fold-training subset.

Only clinical and fusion logistic regressions were refitted for the approved correction. Upstream
W3, W4, W5, CNN, radiomics, and LightGBM outputs were unchanged.


In [ ]:
def fit_fusion_fold(train_frame, validation_frame, test_frame):
    feature_names = list(FULL_FUSION_FEATURES)
    X_train = train_frame[feature_names].to_numpy(dtype=np.float32)
    X_validation = validation_frame[feature_names].to_numpy(dtype=np.float32)
    X_test = test_frame[feature_names].to_numpy(dtype=np.float32)
    y_train = train_frame["label"].to_numpy()

    for matrix in (X_train, X_validation, X_test):
        matrix[matrix <= -1] = np.nan
    imputer = SimpleImputer(strategy="median").fit(X_train)
    X_train = imputer.transform(X_train)
    X_validation = imputer.transform(X_validation)
    X_test = imputer.transform(X_test)
    scaler = StandardScaler().fit(X_train)
    X_train = scaler.transform(X_train)
    X_validation = scaler.transform(X_validation)
    X_test = scaler.transform(X_test)
    model = LogisticRegression(
        max_iter=2000,
        C=1.0,
        class_weight="balanced",
        random_state=42,
    )
    model.fit(X_train, y_train)
    return {
        "model": model,
        "imputer": imputer,
        "scaler": scaler,
        "validation_probabilities": model.predict_proba(X_validation),
        "test_probabilities": model.predict_proba(X_test),
    }


def macro_metrics(labels, probabilities):
    probabilities = np.asarray(probabilities, dtype=float)
    predictions = probabilities.argmax(axis=1)
    return {
        "macro_auc": roc_auc_score(labels, probabilities, multi_class="ovr", average="macro"),
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
    }


## Released aggregate results

The corrected primary full-fusion held-out macro-AUC is 0.945452 (95% CI 0.889577–0.984331),
with 45/56 correct, accuracy 0.803571, and macro-F1 0.804669. Historical full-fusion AUC
0.949675 is retained only as provenance sensitivity. Released tables contain no patient rows.


In [ ]:
aggregate_root = REPO_ROOT / "results" / "aggregate"
paths = {
    "performance": aggregate_root / "internal_performance.csv",
    "sex_sensitivity": aggregate_root / "internal_sex_sensitivity.csv",
    "per_class": aggregate_root / "internal_per_class.csv",
    "paired_tests": aggregate_root / "internal_paired_tests.csv",
    "delong": aggregate_root / "delong_summary.csv",
}
for name, required in paths.items():
    if not required.is_file():
        raise FileNotFoundError(f"Released aggregate artifact is unavailable: {name}")

internal_performance = pd.read_csv(paths["performance"])
sex_sensitivity = pd.read_csv(paths["sex_sensitivity"])
per_class = pd.read_csv(paths["per_class"])
paired_tests = pd.read_csv(paths["paired_tests"])
delong_summary = pd.read_csv(paths["delong"])
internal_performance


## Corrected statistical interpretation

Nine exploratory per-class DeLong comparisons were evaluated. The smallest raw p-value was
0.029347 for full fusion versus W4 in cHCC-CCA; Bonferroni and Benjamini–Hochberg values were
both 0.264126. Full fusion versus W5 in that class had p=0.345737. No comparison survived either
adjustment.

Five-fold paired t-test and exact Wilcoxon results are released separately. Their fold count is
small, so they remain exploratory. Patient-level predictions are intentionally absent.


In [ ]:
required_delong_columns = {
    "comparison", "class", "uncorrected_p", "bonferroni_p_9", "bh_p_9"
}
missing_delong_columns = sorted(required_delong_columns.difference(delong_summary.columns))
if missing_delong_columns:
    raise ValueError(f"DeLong aggregate schema is incomplete: {missing_delong_columns}")
if len(delong_summary) != 9 or len(paired_tests) != 5:
    raise ValueError("Corrected statistical aggregate row counts are inconsistent.")
{
    "sex_sensitivity": sex_sensitivity,
    "per_class": per_class,
    "paired_tests": paired_tests,
    "delong": delong_summary.sort_values("uncorrected_p").reset_index(drop=True),
}
